In [ ]:
#Setup for Target Data:
import pandas as pd
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive', force_remount=True)

# Load PM2.5 target dataset
pm25_path = "/content/drive/MyDrive/ResearchPG/Datasets/Daily_Data.csv"
pm25 = pd.read_csv(pm25_path)

# Check columns
print("Columns:", pm25.columns.tolist())

# Convert date column
pm25["Date"] = pd.to_datetime(pm25["Date"])

# Select Santa Cruz site
target_site_id = 60870007
target_site = pm25[pm25["Site ID"] == target_site_id].copy()

# Get latitude and longitude from the correct columns
site_lat = target_site["Site Latitude"].iloc[0]
site_lon = target_site["Site Longitude"].iloc[0]

print("Target Site ID:", target_site_id)
print("Latitude:", site_lat)
print("Longitude:", site_lon)

# Keep only needed columns
target_site = target_site[
    ["Date", "Site ID", "Site Latitude", "Site Longitude", "Daily Mean PM2.5 Concentration"]
].copy()

target_site = target_site.rename(columns={
    "Site Latitude": "Latitude",
    "Site Longitude": "Longitude",
    "Daily Mean PM2.5 Concentration": "pm25"
})

print(target_site.head())
print("Total rows:", len(target_site))

Mounted at /content/drive
Columns: ['Date', 'Source', 'Site ID', 'POC', 'Daily Mean PM2.5 Concentration', 'Units', 'Daily AQI Value', 'Local Site Name', 'Daily Obs Count', 'Percent Complete', 'AQS Parameter Code', 'AQS Parameter Description', 'Method Code', 'Method Description', 'CBSA Code', 'CBSA Name', 'State FIPS Code', 'State', 'County FIPS Code', 'County', 'Site Latitude', 'Site Longitude']
Target Site ID: 60870007
Latitude: 36.98332
Longitude: -121.98822
        Date   Site ID  Latitude  Longitude  pm25
0 2025-01-01  60870007  36.98332 -121.98822   7.3
1 2025-01-02  60870007  36.98332 -121.98822   6.2
2 2025-01-03  60870007  36.98332 -121.98822   5.6
3 2025-01-04  60870007  36.98332 -121.98822   5.5
4 2025-01-05  60870007  36.98332 -121.98822   7.2
Total rows: 360


In [ ]:
#Authenticate the Earth Engine API:
import ee
import geemap

ee.Authenticate()
ee.Initialize(project='earth-engine-api-492523') #Add your project ID in the quotes.

In [ ]:
#Create a PM2.5 vegetation region around the site and load MODIS NDVI (Google Earth Engine's Vegetation Data)

#Create a point from the PM2.5 site coordinates:
site_point = ee.Geometry.Point([site_lon, site_lat])

#Create a region around the site
#50000 meters = 50 km buffer
region = site_point.buffer(50000).bounds()

#Load MODIS NDVI for 2025 (or whatever your dates are for your dataset:)
modis = (
    ee.ImageCollection("MODIS/061/MOD13Q1")
    .filterDate("2025-01-01", "2026-01-01")
    .select("NDVI")
)

In [ ]:
#Export the vegetation data as a CSV time series:
def image_to_feature(img):
    stats = img.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=region,
        scale=250,
        maxPixels=1e13
    )
    return ee.Feature(None, {
        "date": ee.Date(img.get("system:time_start")).format("YYYY-MM-dd"),
        "NDVI": stats.get("NDVI")
    })

ndvi_fc = modis.map(image_to_feature)

task = ee.batch.Export.table.toDrive(
    collection=ndvi_fc,
    description="NDVI_timeseries",
    folder="EarthEngineExports",
    fileFormat="CSV"
)

task.start()
print("Export started!")

Export started!


In [ ]:
#Check export status:
print(task.status())

{'state': 'READY', 'description': 'NDVI_timeseries', 'priority': 100, 'creation_timestamp_ms': 1786922540668, 'update_timestamp_ms': 1786922540668, 'start_timestamp_ms': 0, 'task_type': 'EXPORT_FEATURES', 'id': 'U2WBPSYXVRJPFY3UVMRMCSWS', 'name': 'projects/earth-engine-api-492523/operations/U2WBPSYXVRJPFY3UVMRMCSWS'}


In [ ]:
#Load the exported NDVI CSV from Drive:
import pandas as pd

#Change this path if needed:
ndvi_path = "/content/drive/MyDrive/EarthEngineExports/NDVI_timeseries.csv"

ndvi = pd.read_csv(ndvi_path)
print(ndvi.head())
print(ndvi.columns)
print("Original rows:", len(ndvi))

  system:index         NDVI        date  \
0   2025_01_01  5852.318122  2025-01-01   
1   2025_01_17  5819.383448  2025-01-17   
2   2025_02_02  5947.415988  2025-02-02   
3   2025_02_18  5941.659118  2025-02-18   
4   2025_03_06  6319.327080  2025-03-06   

                                     .geo  
0  {"type":"MultiPoint","coordinates":[]}  
1  {"type":"MultiPoint","coordinates":[]}  
2  {"type":"MultiPoint","coordinates":[]}  
3  {"type":"MultiPoint","coordinates":[]}  
4  {"type":"MultiPoint","coordinates":[]}  
Index(['system:index', 'NDVI', 'date', '.geo'], dtype='object')
Original rows: 23


In [ ]:
print(ndvi.columns)
print(ndvi.head())

Index(['system:index', 'NDVI', 'date', '.geo'], dtype='object')
  system:index         NDVI        date  \
0   2025_01_01  5852.318122  2025-01-01   
1   2025_01_17  5819.383448  2025-01-17   
2   2025_02_02  5947.415988  2025-02-02   
3   2025_02_18  5941.659118  2025-02-18   
4   2025_03_06  6319.327080  2025-03-06   

                                     .geo  
0  {"type":"MultiPoint","coordinates":[]}  
1  {"type":"MultiPoint","coordinates":[]}  
2  {"type":"MultiPoint","coordinates":[]}  
3  {"type":"MultiPoint","coordinates":[]}  
4  {"type":"MultiPoint","coordinates":[]}  


In [ ]:
# Convert all the NDVI data to daily data for 2025

# Convert the date column
ndvi["date"] = pd.to_datetime(ndvi["date"])

# Keep only the columns you actually need
ndvi = ndvi[["date", "NDVI"]].copy()

# Convert NDVI to numeric
ndvi["NDVI"] = pd.to_numeric(ndvi["NDVI"], errors="coerce")

# Set as index
ndvi = ndvi.set_index("date")

# Get the full daily date range for 2025
full_dates = pd.date_range(start="2025-01-01", end="2025-12-31")
ndvi = ndvi.reindex(full_dates)

# Rename the index
ndvi.index.name = "date"

# Insert missing values
ndvi_daily = ndvi.interpolate(method="linear")

# Reset the index
ndvi_daily = ndvi_daily.reset_index()

print(ndvi_daily.head())
print(ndvi_daily.tail())
print("Total rows:", len(ndvi_daily))
print(ndvi_daily.dtypes)

        date         NDVI
0 2025-01-01  5852.318122
1 2025-01-02  5850.259705
2 2025-01-03  5848.201288
3 2025-01-04  5846.142871
4 2025-01-05  5844.084453
          date         NDVI
360 2025-12-27  6264.444437
361 2025-12-28  6264.444437
362 2025-12-29  6264.444437
363 2025-12-30  6264.444437
364 2025-12-31  6264.444437
Total rows: 365
date    datetime64[ns]
NDVI           float64
dtype: object


In [ ]:
#Save the daily vegetation data:
ndvi_daily_path = "/content/drive/MyDrive/ResearchPG/Datasets/NDVI_daily.csv"
ndvi_daily.to_csv(ndvi_daily_path, index=False)

print("Saved daily NDVI to:", ndvi_daily_path)

Saved daily NDVI to: /content/drive/MyDrive/ResearchPG/Datasets/NDVI_daily.csv


In [ ]:
#Merge the target PM2.5 data with the daily vegetation data:

#When you download your data, make sure your dates match!
target_site["Date"] = pd.to_datetime(target_site["Date"])
ndvi_daily["date"] = pd.to_datetime(ndvi_daily["date"])

#Merge the two datasets:
merged_pm25_ndvi = target_site.merge(
    ndvi_daily,
    left_on = "Date",
    right_on = "date",
    how = "left"

)

#Drop duplicate date column (this is optional, but for the sake of this project, I will do it:)
merged_pm25_ndvi = merged_pm25_ndvi.drop(columns=["date"])

print(merged_pm25_ndvi.head())
print("Merged rows:", len(merged_pm25_ndvi))

        Date   Site ID  Latitude  Longitude  pm25         NDVI
0 2025-01-01  60870007  36.98332 -121.98822   7.3  5852.318122
1 2025-01-02  60870007  36.98332 -121.98822   6.2  5850.259705
2 2025-01-03  60870007  36.98332 -121.98822   5.6  5848.201288
3 2025-01-04  60870007  36.98332 -121.98822   5.5  5846.142871
4 2025-01-05  60870007  36.98332 -121.98822   7.2  5844.084453
Merged rows: 360


In [ ]:
#Save the merged PM2.5 and the vegetation dataset:

merged_path = "/content/drive/MyDrive/ResearchPG/Datasets/PM25_NDVI_merged.csv"
merged_pm25_ndvi.to_csv(merged_path, index=False)

print("Saved merged data to:", merged_path)

Saved merged data to: /content/drive/MyDrive/ResearchPG/Datasets/PM25_NDVI_merged.csv


In [ ]:
#Pull in hourly data for HRRR, which is our weather variables. The first step is to install Herbie and weather dependencies for HRRR
!pip install -q herbie-data==2024.3.0 pandas==2.2.2 cfgrib xarray netcdf4

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.1/56.1 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.2/62.2 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.1/49.1 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 44.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.6/91.6 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.8/11.8 MB 70.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 61.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 424.4/424.4 kB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.8/17.8 MB 59.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 307.5/307.5 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 58.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 62.4 MB/s eta 0:00:00


In [ ]:
#Check the version of pandas (just in case!)

import pandas as pd
print(pd.__version__)

2.2.2


In [ ]:
#Load both Wildfire CSVs

import pandas as pd

fire_j1_path = "/content/drive/MyDrive/ResearchPG/Datasets/WildfireData_J1 VIIRS C2 - fire_archive_J1V-C2_729304.csv"
fire_suomi_path = "/content/drive/MyDrive/ResearchPG/Datasets/Wildfire_Data_SUOMI VIIRS C2 - fire_archive_SV-C2_729305.csv"

fire_j1 = pd.read_csv(fire_j1_path)
fire_suomi = pd.read_csv(fire_suomi_path)

print("J1 columns:", fire_j1.columns.tolist())
print("SUOMI columns:", fire_suomi.columns.tolist())
print("J1 rows:", len(fire_j1))
print("SUOMI rows:", len(fire_suomi))

J1 columns: ['latitude', 'longitude', 'brightness', 'scan', 'track', 'acq_date', 'acq_time', 'satellite', 'instrument', 'confidence', 'version', 'bright_t31', 'frp', 'daynight', 'type']
SUOMI columns: ['latitude', 'longitude', 'brightness', 'scan', 'track', 'acq_date', 'acq_time', 'satellite', 'instrument', 'confidence', 'version', 'bright_t31', 'frp', 'daynight', 'type']
J1 rows: 28619
SUOMI rows: 27808


In [ ]:
#Combine the two datasets

fire_j1["source_name"] = "J1"
fire_suomi["source_name"] = "SUOMI"

fire_all = pd.concat([fire_j1, fire_suomi], ignore_index=True)

print(fire_all.head())
print("Total wildfire rows:", len(fire_all))

   latitude  longitude  brightness  scan  track    acq_date  acq_time  \
0  40.56269 -123.69317      324.13  0.39   0.36  2025-03-01      1012   
1  40.56345 -123.69770      319.53  0.39   0.36  2025-03-01      1012   
2  39.68690 -121.85736      295.60  0.43   0.38  2025-03-01      1012   
3  39.89914 -123.30003      308.64  0.40   0.37  2025-03-01      1012   
4  39.21783 -122.30177      301.19  0.43   0.38  2025-03-01      1013   

  satellite instrument confidence  version  bright_t31   frp daynight  type  \
0       N20      VIIRS          n        2      281.59  3.24        N     0   
1       N20      VIIRS          n        2      277.94  2.23        N     0   
2       N20      VIIRS          n        2      280.26  0.43        N     0   
3       N20      VIIRS          n        2      280.30  0.89        N     0   
4       N20      VIIRS          n        2      279.93  1.00        N     0   

  source_name  
0          J1  
1          J1  
2          J1  
3          J1  
4     

In [ ]:
#Clean the wildfire date and time columns
fire_all["acq_date"] = pd.to_datetime(fire_all["acq_date"])
fire_all["acq_time"] = fire_all["acq_time"].astype(str).str.zfill(4)

fire_all["datetime"] = pd.to_datetime(
    fire_all["acq_date"].dt.strftime("%Y-%m-%d") + " " +
    fire_all["acq_time"].str[:2] + ":" + fire_all["acq_time"].str[2:],
    errors="coerce"
)

print(fire_all[["acq_date", "acq_time", "datetime"]].head())

    acq_date acq_time            datetime
0 2025-03-01     1012 2025-03-01 10:12:00
1 2025-03-01     1012 2025-03-01 10:12:00
2 2025-03-01     1012 2025-03-01 10:12:00
3 2025-03-01     1012 2025-03-01 10:12:00
4 2025-03-01     1013 2025-03-01 10:13:00


In [ ]:
#Filter all Wildfire data near the site:
import numpy as np

def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))

fire_all["distance_km"] = haversine(
    site_lat,
    site_lon,
    fire_all["latitude"],
    fire_all["longitude"]
)

fire_nearby = fire_all[fire_all["distance_km"] <= 100].copy()

print(fire_nearby.head())
print("Nearby wildfire rows:", len(fire_nearby))

    latitude  longitude  brightness  scan  track   acq_date acq_time  \
12  37.75710 -121.65826      296.61  0.47   0.39 2025-03-01     1013   
13  37.45664 -121.93284      302.71  0.46   0.39 2025-03-01     1013   
63  37.50398 -121.08484      301.50  0.38   0.36 2025-03-02     0954   
73  36.41363 -121.38177      332.63  0.40   0.37 2025-03-03     2056   
75  36.44277 -121.35264      330.65  0.40   0.37 2025-03-03     2056   

   satellite instrument confidence  version  bright_t31    frp daynight  type  \
12       N20      VIIRS          n        2      282.57   0.62        N     0   
13       N20      VIIRS          n        2      278.27   0.54        N     2   
63       N20      VIIRS          n        2      281.18   0.49        N     0   
73       N20      VIIRS          n        2      296.18  19.44        D     0   
75       N20      VIIRS          n        2      294.85   5.73        D     0   

   source_name            datetime  distance_km  
12          J1 2025-03-01 10:1

In [ ]:
#Aggregate it to daily features
fire_daily = fire_nearby.groupby("acq_date").agg(
    fire_count=("acq_date", "size"),
    mean_frp=("frp", "mean"),
    max_frp=("frp", "max")
).reset_index()

fire_daily = fire_daily.rename(columns={"acq_date": "Date"})

print(fire_daily.head())
print("Daily wildfire rows:", len(fire_daily))

        Date  fire_count  mean_frp  max_frp
0 2025-03-01           5  0.684000     1.10
1 2025-03-02           2  0.645000     0.80
2 2025-03-03           6  5.673333    19.44
3 2025-03-04           2  4.535000     4.97
4 2025-03-07           6  0.740000     1.57
Daily wildfire rows: 232


In [ ]:
#Force all Wildfire data to 2025
full_dates = pd.date_range(start="2025-01-01", end="2025-12-31")

fire_daily = fire_daily.set_index("Date").reindex(full_dates)
fire_daily.index.name = "Date"

fire_daily["fire_count"] = fire_daily["fire_count"].fillna(0)
fire_daily["mean_frp"] = fire_daily["mean_frp"].fillna(0)
fire_daily["max_frp"] = fire_daily["max_frp"].fillna(0)

fire_daily = fire_daily.reset_index()

print(fire_daily.head())
print(fire_daily.tail())
print("Total wildfire daily rows:", len(fire_daily))

        Date  fire_count  mean_frp  max_frp
0 2025-01-01         0.0       0.0      0.0
1 2025-01-02         0.0       0.0      0.0
2 2025-01-03         0.0       0.0      0.0
3 2025-01-04         0.0       0.0      0.0
4 2025-01-05         0.0       0.0      0.0
          Date  fire_count  mean_frp  max_frp
360 2025-12-27         1.0    4.7000     4.70
361 2025-12-28         2.0    0.4600     0.53
362 2025-12-29         8.0    2.2050     6.68
363 2025-12-30         4.0    0.6875     1.02
364 2025-12-31         0.0    0.0000     0.00
Total wildfire daily rows: 365


In [ ]:
#Save the Wildfire Daily Dataset
fire_daily_path = "/content/drive/MyDrive/ResearchPG/Datasets/Wildfire_daily.csv"
fire_daily.to_csv(fire_daily_path, index=False)

print("Saved wildfire daily data to:", fire_daily_path)

Saved wildfire daily data to: /content/drive/MyDrive/ResearchPG/Datasets/Wildfire_daily.csv


In [ ]:
#Merge it into the PM2.5 + NDVI dataset:
merged_pm25_ndvi["Date"] = pd.to_datetime(merged_pm25_ndvi["Date"])
fire_daily["Date"] = pd.to_datetime(fire_daily["Date"])

merged_pm25_ndvi_fire = merged_pm25_ndvi.merge(fire_daily, on="Date", how="left")

print(merged_pm25_ndvi_fire.head())
print("Merged rows:", len(merged_pm25_ndvi_fire))

        Date   Site ID  Latitude  Longitude  pm25         NDVI  fire_count  \
0 2025-01-01  60870007  36.98332 -121.98822   7.3  5852.318122         0.0   
1 2025-01-02  60870007  36.98332 -121.98822   6.2  5850.259705         0.0   
2 2025-01-03  60870007  36.98332 -121.98822   5.6  5848.201288         0.0   
3 2025-01-04  60870007  36.98332 -121.98822   5.5  5846.142871         0.0   
4 2025-01-05  60870007  36.98332 -121.98822   7.2  5844.084453         0.0   

   mean_frp  max_frp  
0       0.0      0.0  
1       0.0      0.0  
2       0.0      0.0  
3       0.0      0.0  
4       0.0      0.0  
Merged rows: 360


In [ ]:
#Save PM2.5 + NDVI + Wildfire merged dataset:
merged_fire_path = "/content/drive/MyDrive/ResearchPG/Datasets/PM25_NDVI_Wildfire_merged.csv"
merged_pm25_ndvi_fire.to_csv(merged_fire_path, index=False)

print("Saved merged dataset to:", merged_fire_path)

Saved merged dataset to: /content/drive/MyDrive/ResearchPG/Datasets/PM25_NDVI_Wildfire_merged.csv


In [ ]:
#HRRR Setup
from herbie import Herbie
import pandas as pd
import numpy as np
import xarray as xr
from tqdm import tqdm
import os

#Make sure all of these dates are in datetime!
merged_pm25_ndvi_fire["Date"] = pd.to_datetime(merged_pm25_ndvi_fire["Date"])

#These will come from your first cell!
print("Using site latitude:", site_lat)
print("Using site longitude", site_lon)
print("Date range:", merged_pm25_ndvi_fire["Date"].min(), "to", merged_pm25_ndvi_fire["Date"].max())

 ╭─▌▌Herbie─────────────────────────────────────────────╮
 │ INFO: Created a default config file.                 │
 │ You may view/edit Herbie's configuration here:       │
 │          /root/.config/herbie/config.toml            │
 ╰──────────────────────────────────────────────────────╯

Using site latitude: 36.98332
Using site longitude -121.98822
Date range: 2025-01-01 00:00:00 to 2025-12-31 00:00:00


In [ ]:
# Function to download HRRR weather variables for one day

def get_hrrr_for_date(date, site_lat, site_lon):
    """
    Pulls HRRR surface weather data for one date near the PM2.5 site.
    Uses the 12z run because it gives a consistent daily snapshot.
    """

    try:
        # HRRR analysis run for that date at 12:00 UTC
        h = Herbie(
            pd.to_datetime(date).strftime("%Y-%m-%d 12:00"),
            model="hrrr",
            product="sfc",
            fxx=0
        )

        # Pull common surface weather variables
        ds = h.xarray(
            ":(TMP|DPT):2 m|:(UGRD|VGRD):10 m|:GUST:surface|:PRES:surface"
        )

        # Select nearest grid point to your PM2.5 site
        point = ds.herbie.nearest_points(points=[(site_lon, site_lat)])

        row = {
            "Date": pd.to_datetime(date),
        }

        # Extract variables safely
        for var in point.data_vars:
            value = point[var].values

            if np.size(value) > 0:
                row[var] = float(np.ravel(value)[0])

        return row

    except Exception as e:
        print(f"Could not get HRRR for {date}: {e}")
        return {
            "Date": pd.to_datetime(date),
            "hrrr_error": str(e)
        }

In [ ]:
#HRRR data every 5th day because every day will overload the Colab
#Then, later, we can interpolate this data back to daily data.

merged_pm25_ndvi_fire["Date"] = pd.to_datetime(merged_pm25_ndvi_fire["Date"])

all_dates = sorted(merged_pm25_ndvi_fire["Date"].dropna().unique())

#You can change this number. I am using 5, but you can change it to 3 or 4 for more accuracy or to 7 if Colab crashes.
sample_step = 5

dates = all_dates[::sample_step]

print("Original daily data:", len(all_dates))
print("HRRR sampled data:", len(dates))
print("First sampled date:", dates[0])
print("Last sampled date:", dates[-1])

Original daily data: 360
HRRR sampled data: 72
First sampled date: 2025-01-01 00:00:00
Last sampled date: 2025-12-27 00:00:00


In [ ]:
#Lightweight HRRR pull function
# This avoids using xarray, which was causing memory crashes.
# It only downloads a very small subset of HRRR data (2m temperature).

import os
import gc
import pandas as pd
import numpy as np
from herbie import Herbie

def get_hrrr_minimal(date):
    """
    Downloads a very small HRRR subset for a single date.
    Does not load the full dataset into memory.
    """

    row = {"Date": pd.to_datetime(date)}

    try:
        h = Herbie(
            pd.to_datetime(date).strftime("%Y-%m-%d 12:00"),
            model="hrrr",
            product="sfc",
            fxx=0
        )

        # Download only temperature at 2 meters above ground
        grib_file = h.download(searchString=":TMP:2 m above ground:")

        # At this stage, we are only confirming the file downloads successfully
        # We are not extracting values yet to avoid memory issues
        row["temp_2m_K"] = np.nan

        # Immediately delete file to prevent disk/memory buildup
        if os.path.exists(grib_file):
            os.remove(grib_file)

        gc.collect()

    except Exception as e:
        print("Failed:", date, e)
        row["temp_2m_K"] = np.nan

    return row

In [ ]:
# Controlled loop to test HRRR downloads safely
# This runs on only a few dates to confirm stability before scaling up.

from tqdm import tqdm

hrrr_rows = []

# Use only a few dates to prevent crashing during testing
test_dates = dates[:3]

for date in tqdm(test_dates):
    row = get_hrrr_minimal(date)
    hrrr_rows.append(row)

    # Save progress after each iteration
    pd.DataFrame(hrrr_rows).to_csv(
        "/content/drive/MyDrive/ResearchPG/Datasets/HRRR_partial_sampled.csv",
        index=False
    )

    print("Saved:", date)

    gc.collect()

hrrr_sampled = pd.DataFrame(hrrr_rows)

print("Completed test run")
print(hrrr_sampled)

  0%|          | 0/3 [00:00<?, ?it/s]

✅ Found ┊ model=hrrr ┊ product=sfc ┊ 2025-Jan-01 12:00 UTC F00 ┊ GRIB2 @ aws ┊ IDX @ aws
👨🏻‍🏭 Created directory: [/root/data/hrrr/20250101]


 33%|███▎      | 1/3 [00:02<00:05,  2.57s/it]

Saved: 2025-01-01 00:00:00
✅ Found ┊ model=hrrr ┊ product=sfc ┊ 2025-Jan-06 12:00 UTC F00 ┊ GRIB2 @ aws ┊ IDX @ aws
👨🏻‍🏭 Created directory: [/root/data/hrrr/20250106]


 67%|██████▋   | 2/3 [00:04<00:02,  2.18s/it]

Saved: 2025-01-06 00:00:00
✅ Found ┊ model=hrrr ┊ product=sfc ┊ 2025-Jan-11 12:00 UTC F00 ┊ GRIB2 @ aws ┊ IDX @ aws
👨🏻‍🏭 Created directory: [/root/data/hrrr/20250111]


100%|██████████| 3/3 [00:06<00:00,  2.25s/it]

Saved: 2025-01-11 00:00:00
Completed test run
        Date  temp_2m_K
0 2025-01-01        NaN
1 2025-01-06        NaN
2 2025-01-11        NaN


In [ ]:
# Function to extract one site value from a small HRRR GRIB file
# This uses xarray only after downloading one small HRRR variable subset.
# This is much safer than loading multiple HRRR variables at once.

import xarray as xr
import numpy as np
import gc
import os

def extract_point_from_grib(grib_file, site_lat, site_lon):
    """
    Opens a small HRRR GRIB file and extracts the nearest grid value
    to the PM2.5 monitoring site.
    """

    ds = None

    try:
        ds = xr.open_dataset(
            grib_file,
            engine="cfgrib",
            backend_kwargs={"indexpath": ""}
        )

        data_vars = list(ds.data_vars)

        if len(data_vars) == 0:
            return np.nan

        var_name = data_vars[0]

        lats = ds["latitude"].values
        lons = ds["longitude"].values

        target_lon = site_lon

        # HRRR longitude may use 0 to 360 instead of -180 to 180
        if np.nanmax(lons) > 180 and target_lon < 0:
            target_lon = target_lon + 360

        distance = (lats - site_lat) ** 2 + (lons - target_lon) ** 2
        y_index, x_index = np.unravel_index(np.nanargmin(distance), distance.shape)

        value = ds[var_name].values[y_index, x_index]

        return float(value)

    except Exception as e:
        print("Extraction failed:", e)
        return np.nan

    finally:
        if ds is not None:
            ds.close()

        gc.collect()

In [ ]:
#Function to extract multiple HRRR weather features for one date
# Each variable is downloaded separately to reduce memory pressure.

hrrr_variable_map = {
    "temp_2m_K": ":TMP:2 m above ground:",
    "dewpoint_2m_K": ":DPT:2 m above ground:",
    "u_wind_10m": ":UGRD:10 m above ground:",
    "v_wind_10m": ":VGRD:10 m above ground:"
}

def get_hrrr_features_for_date(date, site_lat, site_lon):
    """
    Downloads and extracts HRRR weather variables for one date.
    Saves only the nearest point value for the site.
    """

    row = {"Date": pd.to_datetime(date)}

    try:
        h = Herbie(
            pd.to_datetime(date).strftime("%Y-%m-%d 12:00"),
            model="hrrr",
            product="sfc",
            fxx=0
        )

        for feature_name, search_string in hrrr_variable_map.items():
            grib_file = None

            try:
                grib_file = h.download(searchString=search_string)

                value = extract_point_from_grib(
                    grib_file=grib_file,
                    site_lat=site_lat,
                    site_lon=site_lon
                )

                row[feature_name] = value

            except Exception as e:
                print("Failed variable:", feature_name, "for", date, e)
                row[feature_name] = np.nan

            finally:
                if grib_file is not None and os.path.exists(grib_file):
                    os.remove(grib_file)

                gc.collect()

    except Exception as e:
        print("Failed date:", date, e)
        row["hrrr_error"] = str(e)

    return row

In [ ]:
# Pull sampled HRRR weather values
# This uses the sampled dates from Cell 24.
# Start with sample_step = 15 in Cell 24.
# If this works, you can later try sample_step = 10 or 7.

hrrr_extracted_path = "/content/drive/MyDrive/ResearchPG/Datasets/HRRR_extracted_sampled.csv"

hrrr_rows = []

for date in tqdm(dates):
    row = get_hrrr_features_for_date(date, site_lat, site_lon)
    hrrr_rows.append(row)

    # Save after every date so progress is not lost
    pd.DataFrame(hrrr_rows).to_csv(hrrr_extracted_path, index=False)

    print("Saved HRRR values for:", pd.to_datetime(date).strftime("%Y-%m-%d"))

    gc.collect()

hrrr_sampled = pd.DataFrame(hrrr_rows)

print("Completed sampled HRRR extraction")
print("Rows:", len(hrrr_sampled))
print("Columns:", hrrr_sampled.columns.tolist())

hrrr_sampled.head()

  0%|          | 0/72 [00:00<?, ?it/s]

✅ Found ┊ model=hrrr ┊ product=sfc ┊ 2025-Jan-01 12:00 UTC F00 ┊ GRIB2 @ aws ┊ IDX @ aws


  1%|▏         | 1/72 [00:13<15:27, 13.06s/it]

Saved HRRR values for: 2025-01-01
✅ Found ┊ model=hrrr ┊ product=sfc ┊ 2025-Jan-06 12:00 UTC F00 ┊ GRIB2 @ aws ┊ IDX @ aws


  3%|▎         | 2/72 [00:19<10:44,  9.20s/it]

...

 99%|█████████▊| 71/72 [05:57<00:04,  4.59s/it]

Saved HRRR values for: 2025-12-22
✅ Found ┊ model=hrrr ┊ product=sfc ┊ 2025-Dec-27 12:00 UTC F00 ┊ GRIB2 @ aws ┊ IDX @ aws
👨🏻‍🏭 Created directory: [/root/data/hrrr/20251227]


100%|██████████| 72/72 [06:03<00:00,  5.04s/it]

Saved HRRR values for: 2025-12-27
Completed sampled HRRR extraction
Rows: 72
Columns: ['Date', 'temp_2m_K', 'dewpoint_2m_K', 'u_wind_10m', 'v_wind_10m']


,Date,temp_2m_K,dewpoint_2m_K,u_wind_10m,v_wind_10m
0,2025-01-01,278.281555,271.310364,-0.777533,-1.541708
1,2025-01-06,280.569580,277.245850,-1.258052,-1.107056
2,2025-01-11,279.896637,277.852112,-2.612053,-0.625494
3,2025-01-16,278.697449,270.374786,-1.154705,-1.681899
4,2025-01-21,278.448914,268.345551,-0.384708,-2.712133


In [ ]:
# Clean HRRR features and create useful weather variables

hrrr_daily_clean = hrrr_sampled.copy()

hrrr_daily_clean["Date"] = pd.to_datetime(hrrr_daily_clean["Date"])

if "hrrr_error" in hrrr_daily_clean.columns:
    hrrr_daily_clean = hrrr_daily_clean.drop(columns=["hrrr_error"])

# Convert Kelvin to Celsius
if "temp_2m_K" in hrrr_daily_clean.columns:
    hrrr_daily_clean["temp_2m_C"] = hrrr_daily_clean["temp_2m_K"] - 273.15

if "dewpoint_2m_K" in hrrr_daily_clean.columns:
    hrrr_daily_clean["dewpoint_2m_C"] = hrrr_daily_clean["dewpoint_2m_K"] - 273.15

# Calculate wind speed from u and v wind components
if "u_wind_10m" in hrrr_daily_clean.columns and "v_wind_10m" in hrrr_daily_clean.columns:
    hrrr_daily_clean["wind_speed_10m"] = np.sqrt(
        hrrr_daily_clean["u_wind_10m"] ** 2 + hrrr_daily_clean["v_wind_10m"] ** 2
    )

print("Cleaned HRRR columns:")
print(hrrr_daily_clean.columns.tolist())

hrrr_daily_clean.head()

Cleaned HRRR columns:
['Date', 'temp_2m_K', 'dewpoint_2m_K', 'u_wind_10m', 'v_wind_10m', 'temp_2m_C', 'dewpoint_2m_C', 'wind_speed_10m']


,Date,temp_2m_K,dewpoint_2m_K,u_wind_10m,v_wind_10m,temp_2m_C,dewpoint_2m_C,wind_speed_10m
0,2025-01-01,278.281555,271.310364,-0.777533,-1.541708,5.131555,-1.839636,1.726679
1,2025-01-06,280.569580,277.245850,-1.258052,-1.107056,7.419580,4.095850,1.675788
2,2025-01-11,279.896637,277.852112,-2.612053,-0.625494,6.746637,4.702112,2.685901
3,2025-01-16,278.697449,270.374786,-1.154705,-1.681899,5.547449,-2.775214,2.040129
4,2025-01-21,278.448914,268.345551,-0.384708,-2.712133,5.298914,-4.804449,2.739282


In [ ]:
#Convert sampled HRRR data back into daily data
# Your main PM2.5 dataset is already daily.
# This fills HRRR values for the missing days between sampled dates.

full_date_range = pd.DataFrame({
    "Date": pd.date_range(
        start=merged_pm25_ndvi_fire["Date"].min(),
        end=merged_pm25_ndvi_fire["Date"].max(),
        freq="D"
    )
})

hrrr_daily_filled = full_date_range.merge(
    hrrr_daily_clean,
    on="Date",
    how="left"
)

numeric_cols = hrrr_daily_filled.select_dtypes(include="number").columns

hrrr_daily_filled[numeric_cols] = hrrr_daily_filled[numeric_cols].interpolate(
    method="linear"
)

hrrr_daily_filled[numeric_cols] = hrrr_daily_filled[numeric_cols].bfill().ffill()

print("Daily HRRR rows:", len(hrrr_daily_filled))
print("Missing values after filling:")
print(hrrr_daily_filled.isna().sum())

hrrr_daily_filled.head()

Daily HRRR rows: 365
Missing values after filling:
Date              0
temp_2m_K         0
dewpoint_2m_K     0
u_wind_10m        0
v_wind_10m        0
temp_2m_C         0
dewpoint_2m_C     0
wind_speed_10m    0
dtype: int64


,Date,temp_2m_K,dewpoint_2m_K,u_wind_10m,v_wind_10m,temp_2m_C,dewpoint_2m_C,wind_speed_10m
0,2025-01-01,278.281555,271.310364,-0.777533,-1.541708,5.131555,-1.839636,1.726679
1,2025-01-02,278.739160,272.497461,-0.873636,-1.454778,5.589160,-0.652539,1.716501
2,2025-01-03,279.196765,273.684558,-0.969740,-1.367847,6.046765,0.534558,1.706323
3,2025-01-04,279.654370,274.871655,-1.065844,-1.280917,6.504370,1.721655,1.696145
4,2025-01-05,280.111975,276.058752,-1.161948,-1.193986,6.961975,2.908752,1.685967


In [ ]:
# Save daily HRRR dataset

hrrr_daily_path = "/content/drive/MyDrive/ResearchPG/Datasets/HRRR_daily_filled.csv"

hrrr_daily_filled.to_csv(hrrr_daily_path, index=False)

print("Saved daily HRRR dataset to:", hrrr_daily_path)

Saved daily HRRR dataset to: /content/drive/MyDrive/ResearchPG/Datasets/HRRR_daily_filled.csv


In [ ]:
#Merge HRRR with PM2.5 + NDVI + wildfire data

merged_pm25_ndvi_fire["Date"] = pd.to_datetime(merged_pm25_ndvi_fire["Date"])
hrrr_daily_filled["Date"] = pd.to_datetime(hrrr_daily_filled["Date"])

final_merged_dataset = merged_pm25_ndvi_fire.merge(
    hrrr_daily_filled,
    on="Date",
    how="left"
)

print("Final dataset shape:", final_merged_dataset.shape)
print("Final dataset columns:")
print(final_merged_dataset.columns.tolist())

final_merged_dataset.head()

Final dataset shape: (360, 16)
Final dataset columns:
['Date', 'Site ID', 'Latitude', 'Longitude', 'pm25', 'NDVI', 'fire_count', 'mean_frp', 'max_frp', 'temp_2m_K', 'dewpoint_2m_K', 'u_wind_10m', 'v_wind_10m', 'temp_2m_C', 'dewpoint_2m_C', 'wind_speed_10m']


,Date,Site ID,Latitude,Longitude,pm25,NDVI,fire_count,mean_frp,max_frp,temp_2m_K,dewpoint_2m_K,u_wind_10m,v_wind_10m,temp_2m_C,dewpoint_2m_C,wind_speed_10m
0,2025-01-01,60870007,36.98332,-121.98822,7.3,5852.318122,0.0,0.0,0.0,278.281555,271.310364,-0.777533,-1.541708,5.131555,-1.839636,1.726679
1,2025-01-02,60870007,36.98332,-121.98822,6.2,5850.259705,0.0,0.0,0.0,278.739160,272.497461,-0.873636,-1.454778,5.589160,-0.652539,1.716501
2,2025-01-03,60870007,36.98332,-121.98822,5.6,5848.201288,0.0,0.0,0.0,279.196765,273.684558,-0.969740,-1.367847,6.046765,0.534558,1.706323
3,2025-01-04,60870007,36.98332,-121.98822,5.5,5846.142871,0.0,0.0,0.0,279.654370,274.871655,-1.065844,-1.280917,6.504370,1.721655,1.696145
4,2025-01-05,60870007,36.98332,-121.98822,7.2,5844.084453,0.0,0.0,0.0,280.111975,276.058752,-1.161948,-1.193986,6.961975,2.908752,1.685967


In [ ]:
# Save final PM2.5 + NDVI + wildfire + HRRR dataset

final_merged_path = "/content/drive/MyDrive/ResearchPG/Datasets/PM25_NDVI_Wildfire_HRRR_merged.csv"

final_merged_dataset.to_csv(final_merged_path, index=False)

print("Saved final merged dataset to:", final_merged_path)

Saved final merged dataset to: /content/drive/MyDrive/ResearchPG/Datasets/PM25_NDVI_Wildfire_HRRR_merged.csv


In [ ]:
import pandas as pd
import numpy as np
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

DATA_DIR = "/content/drive/MyDrive/ResearchPG/Datasets/"
# Files needed in this folder: Daily_Data.csv, NDVI_daily.csv, HRRR_extracted_sampled.csv,
# "WildfireData_J1 VIIRS C2 - fire_archive_J1V-C2_729304.csv",
# "Wildfire_Data_SUOMI VIIRS C2 - fire_archive_SV-C2_729305.csv"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Reload the PM2.5 target (same as above, kept here so this section can run standalone)
pm25 = pd.read_csv(DATA_DIR + "Daily_Data.csv")
pm25["Date"] = pd.to_datetime(pm25["Date"])
target_site_id = 60870007
target_site = pm25[pm25["Site ID"] == target_site_id].copy()
site_lat = target_site["Site Latitude"].iloc[0]
site_lon = target_site["Site Longitude"].iloc[0]
target_site = target_site[["Date","Site ID","Site Latitude","Site Longitude","Daily Mean PM2.5 Concentration"]].copy()
target_site = target_site.rename(columns={"Site Latitude":"Latitude","Site Longitude":"Longitude","Daily Mean PM2.5 Concentration":"pm25"})
print("PM2.5 target rows:", len(target_site), "| site:", site_lat, site_lon)

PM2.5 target rows: 360 | site: 36.98332 -121.98822


In [ ]:
# NDVI: divide by 10,000 -- MOD13Q1's scale factor (verified against the LP DAAC/NASA MOD13
# User Guide). The raw values are stored as integers around 5,800-6,400; this puts them back
# on the standard -1..1 NDVI scale. Interpolation direction is still two-sided here (the raw
# sparse ~23-point export wasn't available to redo it as forward-fill-only).
ndvi_daily = pd.read_csv(DATA_DIR + "NDVI_daily.csv")
ndvi_daily["date"] = pd.to_datetime(ndvi_daily["date"])
ndvi_daily["NDVI"] = ndvi_daily["NDVI"] / 10000.0
print("NDVI corrected range:", ndvi_daily["NDVI"].min(), "to", ndvi_daily["NDVI"].max())

NDVI corrected range: 0.5287838492353812 to 0.6408802322043698


In [ ]:
# Wildfire features, rebuilt from the raw VIIRS archives with three additions:
#   - confidence filtering (drops low-confidence detections)
#   - inverse-distance-weighted FRP (closer fires count more)
#   - total daily FRP and a 3-day cumulative fire-activity feature
def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1; dlon = lon2 - lon1
    a = np.sin(dlat/2.0)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2.0)**2
    return 2*R*np.arcsin(np.sqrt(a))

fire_j1 = pd.read_csv(DATA_DIR + "WildfireData_J1 VIIRS C2 - fire_archive_J1V-C2_729304.csv")
fire_suomi = pd.read_csv(DATA_DIR + "Wildfire_Data_SUOMI VIIRS C2 - fire_archive_SV-C2_729305.csv")
fire_j1["source_name"] = "J1"; fire_suomi["source_name"] = "SUOMI"
fire_all = pd.concat([fire_j1, fire_suomi], ignore_index=True)
fire_all["acq_date"] = pd.to_datetime(fire_all["acq_date"])
fire_all["distance_km"] = haversine(site_lat, site_lon, fire_all["latitude"], fire_all["longitude"])

n_before_conf = len(fire_all[fire_all["distance_km"] <= 100])
fire_nearby = fire_all[(fire_all["distance_km"] <= 100) & (fire_all["confidence"] != "l")].copy()
print(f"Detections within 100km: {n_before_conf} before confidence filter, {len(fire_nearby)} after (dropped {n_before_conf-len(fire_nearby)} low-confidence)")

fire_nearby["idw"] = 1.0 / (1.0 + fire_nearby["distance_km"])
fire_nearby["weighted_frp"] = fire_nearby["frp"] * fire_nearby["idw"]

fire_daily = fire_nearby.groupby("acq_date").agg(
    fire_count=("acq_date","size"), mean_frp=("frp","mean"), max_frp=("frp","max"),
    total_frp=("frp","sum"), distance_weighted_frp=("weighted_frp","sum")
).reset_index().rename(columns={"acq_date":"Date"})

full_dates = pd.date_range(start="2025-01-01", end="2025-12-31")
fire_daily = fire_daily.set_index("Date").reindex(full_dates)
fire_daily.index.name = "Date"
fcols = ["fire_count","mean_frp","max_frp","total_frp","distance_weighted_frp"]
fire_daily[fcols] = fire_daily[fcols].fillna(0)
fire_daily = fire_daily.reset_index()
fire_daily["fire_count_3day_sum"] = fire_daily["fire_count"].rolling(window=3, min_periods=1).sum()
print("Fire feature columns:", fire_daily.columns.tolist())

Detections within 100km: 1040 before confidence filter, 993 after (dropped 47 low-confidence)
Fire feature columns: ['Date', 'fire_count', 'mean_frp', 'max_frp', 'total_frp', 'distance_weighted_frp', 'fire_count_3day_sum']


In [ ]:
# HRRR: forward-fill only for the 293 non-sampled days (no bfill, no two-sided interpolation),
# so a day's weather only ever comes from a pull made on or before that day. Also derives
# relative humidity and wind direction from the fields already pulled -- no new download needed.
hrrr_sampled = pd.read_csv(DATA_DIR + "HRRR_extracted_sampled.csv")
hrrr_sampled["Date"] = pd.to_datetime(hrrr_sampled["Date"])
hrrr_daily_clean = hrrr_sampled.copy()
hrrr_daily_clean["temp_2m_C"] = hrrr_daily_clean["temp_2m_K"] - 273.15
hrrr_daily_clean["dewpoint_2m_C"] = hrrr_daily_clean["dewpoint_2m_K"] - 273.15
hrrr_daily_clean["wind_speed_10m"] = np.sqrt(hrrr_daily_clean["u_wind_10m"]**2 + hrrr_daily_clean["v_wind_10m"]**2)

full_date_range = pd.DataFrame({"Date": pd.date_range(start="2025-01-01", end="2025-12-31", freq="D")})
hrrr_daily = full_date_range.merge(hrrr_daily_clean, on="Date", how="left")
numeric_cols = hrrr_daily.select_dtypes(include="number").columns
hrrr_daily[numeric_cols] = hrrr_daily[numeric_cols].ffill()

# Relative humidity -- Alduchov-Eskridge Magnus-Tetens formula (a=17.625, b=243.04 degC)
def relative_humidity(temp_c, dewpoint_c):
    a, b = 17.625, 243.04
    es_t = np.exp((a*temp_c)/(b+temp_c)); es_td = np.exp((a*dewpoint_c)/(b+dewpoint_c))
    return (100.0 * (es_td/es_t)).clip(0, 100)
hrrr_daily["relative_humidity"] = relative_humidity(hrrr_daily["temp_2m_C"], hrrr_daily["dewpoint_2m_C"])

# Wind direction -- meteorological convention, sin/cos encoded to avoid the 0/360 discontinuity
wind_dir_deg = (270 - np.degrees(np.arctan2(hrrr_daily["v_wind_10m"], hrrr_daily["u_wind_10m"]))) % 360
hrrr_daily["wind_dir_sin"] = np.sin(np.radians(wind_dir_deg))
hrrr_daily["wind_dir_cos"] = np.cos(np.radians(wind_dir_deg))

print("Missing values after fill:", hrrr_daily[numeric_cols].isna().sum().sum())
print("RH sample:", hrrr_daily["relative_humidity"].head(5).round(1).tolist())
print("Wind dir (deg) sample:", wind_dir_deg.head(5).round(1).tolist())

Missing values after fill: 0
RH sample: [60.7, 60.7, 60.7, 60.7, 60.7]
Wind dir (deg) sample: [26.8, 26.8, 26.8, 26.8, 26.8]


In [ ]:
# Boundary layer height, surface pressure, and precipitation -- not pulled yet.
# GRIB field names verified against NOAA's official HRRR table (rapidrefresh.noaa.gov/hrrr/HRRRv4_GRIB2_WRFTWO.txt):
#   HPBL (boundary layer height, m) - record 155
#   PRES (surface pressure, Pa) - record 44
#   APCP (precipitation) - record 84; needs fxx=1 since the fxx=0 analysis has zero
#     accumulation by definition (no time has passed yet)
# Run this cell to pull them, save hrrr_extended_df, then merge it into hrrr_daily above
# (on "Date") and add "hpbl_m", "pres_surface_pa", "apcp_mm" to env_cols two cells down.

import os, gc
from tqdm import tqdm
from herbie import Herbie

extended_variable_map = {
    "hpbl_m":          ":HPBL:surface:",
    "pres_surface_pa": ":PRES:surface:",
}

def get_hrrr_extended_for_date(date, site_lat, site_lon):
    row = {"Date": pd.to_datetime(date)}
    try:
        h0 = Herbie(pd.to_datetime(date).strftime("%Y-%m-%d 12:00"), model="hrrr", product="sfc", fxx=0)
        for feature_name, search_string in extended_variable_map.items():
            grib_file = None
            try:
                grib_file = h0.download(searchString=search_string)
                row[feature_name] = extract_point_from_grib(grib_file, site_lat, site_lon)
            except Exception as e:
                print("Failed variable:", feature_name, "for", date, e)
                row[feature_name] = np.nan
            finally:
                if grib_file is not None and os.path.exists(grib_file):
                    os.remove(grib_file)
                gc.collect()
    except Exception as e:
        print("Failed fxx=0 pull for", date, e)
        row["hpbl_m"] = np.nan
        row["pres_surface_pa"] = np.nan

    try:
        h1 = Herbie(pd.to_datetime(date).strftime("%Y-%m-%d 12:00"), model="hrrr", product="sfc", fxx=1)
        grib_file = h1.download(searchString=":APCP:surface:0-1 hour acc fcst:")
        row["apcp_mm"] = extract_point_from_grib(grib_file, site_lat, site_lon)
        if os.path.exists(grib_file):
            os.remove(grib_file)
        gc.collect()
    except Exception as e:
        print("Failed precip pull for", date, e)
        row["apcp_mm"] = np.nan
    return row

hrrr_extended_rows = []
hrrr_extended_path = DATA_DIR + "HRRR_extracted_extended.csv"
for date in tqdm(dates):   # `dates` = the same 72 sampled dates from the HRRR pull above
    row = get_hrrr_extended_for_date(date, site_lat, site_lon)
    hrrr_extended_rows.append(row)
    pd.DataFrame(hrrr_extended_rows).to_csv(hrrr_extended_path, index=False)
    gc.collect()

hrrr_extended_df = pd.DataFrame(hrrr_extended_rows)
print("Completed extended HRRR extraction")
print("Rows:", len(hrrr_extended_df))
print("Columns:", hrrr_extended_df.columns.tolist())
hrrr_extended_df.head()

  0%|          | 0/72 [00:00<?, ?it/s]

✅ Found ┊ model=hrrr ┊ product=sfc ┊ 2025-Jan-01 12:00 UTC F00 ┊ GRIB2 @ aws ┊ IDX @ aws
✅ Found ┊ model=hrrr ┊ product=sfc ┊ 2025-Jan-01 12:00 UTC F01 ┊ GRIB2 @ aws ┊ IDX @ aws
Extraction failed: [Errno 2] No such file or directory: '/root/data/hrrr/20250101/subset_bce4b1fe__hrrr.t12z.wrfsfcf01.grib2'


  1%|▏         | 1/72 [00:03<04:38,  3.93s/it]

✅ Found ┊ model=hrrr ┊ product=sfc ┊ 2025-Jan-06 12:00 UTC F00 ┊ GRIB2 @ aws ┊ IDX @ aws
✅ Found ┊ model=hrrr ┊ product=sfc ┊ 2025-Jan-06 12:00 UTC F01 ┊ GRIB2 @ aws ┊ IDX @ aws
Extraction failed: [Errno 2] No such file or directory: '/root/data/hrrr/20250106/subset_28e4b1fe__hrrr.t12z.wrfsfcf01.grib2'


  3%|▎         | 2/72 [00:07<04:24,  3.77s/it]

...

✅ Found ┊ model=hrrr ┊ product=sfc ┊ 2025-Apr-06 12:00 UTC F00 ┊ GRIB2 @ aws ┊ IDX @ aws
✅ Found ┊ model=hrrr ┊ product=sfc ┊ 2025-Apr-06 12:00 UTC F01 ┊ GRIB2 @ aws ┊ IDX @ aws
Extraction failed: [Errno 2] No such file or directory: '/root/data/hrrr/20250406/subset_05e4b1fe__hrrr.t12z.wrfsfcf01.grib2'


 28%|██▊       | 20/72 [01:14<03:05,  3.57s/it]

✅ Found ┊ model=hrrr ┊ product=sfc ┊ 2025-Apr-11 12:00 UTC F00 ┊ GRIB2 @ aws ┊ IDX @ aws


 28%|██▊       | 20/72 [01:16<03:17,  3.81s/it]


KeyboardInterrupt: 

In [ ]:
# Merge everything, drop constant/duplicate columns, shift environmental predictors to
# day t-1 (so day t's features only ever use information available before day t), and fix
# the rolling PM2.5 features (shift before rolling, so the window never includes the current
# day's own value).
target_col = "pm25"
merged = target_site.merge(ndvi_daily.rename(columns={"date":"Date"}), on="Date", how="left")
merged = merged.merge(fire_daily, on="Date", how="left")
merged = merged.merge(hrrr_daily, on="Date", how="left")
model_df = merged.sort_values("Date").reset_index(drop=True)

model_df["month"] = model_df["Date"].dt.month
model_df["day_of_year"] = model_df["Date"].dt.dayofyear
model_df["day_of_week"] = model_df["Date"].dt.dayofweek

# Site ID/Latitude/Longitude are constant for a single site (zero information).
# temp_2m_K/dewpoint_2m_K duplicate the Celsius versions. Raw u_wind_10m/v_wind_10m are
# superseded by wind speed + direction.
drop_cols = ["Site ID","Latitude","Longitude","temp_2m_K","dewpoint_2m_K","u_wind_10m","v_wind_10m"]
model_df = model_df.drop(columns=[c for c in drop_cols if c in model_df.columns])

env_cols = ["NDVI","fire_count","mean_frp","max_frp","total_frp","distance_weighted_frp",
            "fire_count_3day_sum","temp_2m_C","dewpoint_2m_C","wind_speed_10m",
            "relative_humidity","wind_dir_sin","wind_dir_cos"]
# once hrrr_extended_df has been merged in above, also add: "hpbl_m", "pres_surface_pa", "apcp_mm"
env_cols = [c for c in env_cols if c in model_df.columns]
for c in env_cols:
    model_df[c] = model_df[c].shift(1)

model_df["pm25_lag_1"] = model_df[target_col].shift(1)
model_df["pm25_lag_3"] = model_df[target_col].shift(3)
model_df["pm25_lag_7"] = model_df[target_col].shift(7)
model_df["pm25_rolling_3"] = model_df[target_col].shift(1).rolling(window=3).mean()
model_df["pm25_rolling_7"] = model_df[target_col].shift(1).rolling(window=7).mean()

exclude_cols = ["Date", target_col]
numeric_cols = model_df.select_dtypes(include=["number"]).columns.tolist()
feature_cols = [c for c in numeric_cols if c not in exclude_cols]
print(f"Final feature set ({len(feature_cols)} features):")
print(feature_cols)

In [ ]:
# Report missing data before dropping anything, then drop (not fill) rows that can't be
# completed causally -- no more blanket interpolate/bfill/ffill across the whole dataset.
ml_df = model_df[["Date", target_col] + feature_cols].copy()
missing_counts = ml_df[feature_cols + [target_col]].isna().sum()
print("Missing-data audit (nonzero only):")
print(missing_counts[missing_counts>0].sort_values(ascending=False))

ml_df = ml_df.dropna(subset=[target_col])
rows_before = len(ml_df)
ml_df = ml_df.dropna().reset_index(drop=True)
print(f"\nRows dropped: {rows_before - len(ml_df)} | Final shape: {ml_df.shape}")

In [ ]:
from sklearn.preprocessing import MinMaxScaler

split_index = int(len(ml_df) * 0.8)
train_df, test_df = ml_df.iloc[:split_index], ml_df.iloc[split_index:]
X_train, y_train = train_df[feature_cols], train_df[target_col]
X_test, y_test = test_df[feature_cols], test_df[target_col]
print("Training rows:", len(train_df), "| Testing rows:", len(test_df))

x_scaler = MinMaxScaler(); X_train_scaled = x_scaler.fit_transform(X_train); X_test_scaled = x_scaler.transform(X_test)
y_scaler = MinMaxScaler()
y_train_scaled = y_scaler.fit_transform(np.array(y_train).reshape(-1,1)).ravel()
y_test_scaled = y_scaler.transform(np.array(y_test).reshape(-1,1)).ravel()

In [ ]:
from sklearn.linear_model import LinearRegression, ElasticNet
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

results = {}; preds = {}
def evaluate(name, y_pred):
    mae = mean_absolute_error(y_test, y_pred); mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse); r2 = r2_score(y_test, y_pred)
    results[name] = dict(MAE=mae, MSE=mse, RMSE=rmse, R2=r2); preds[name] = y_pred
    print(f"{name:35s} MAE={mae:.4f}  RMSE={rmse:.4f}  R2={r2:.4f}")

evaluate("Baseline (persistence)", np.array(test_df["pm25_lag_1"]))
lin = LinearRegression().fit(X_train_scaled, y_train_scaled)
evaluate("Linear Regression", y_scaler.inverse_transform(lin.predict(X_test_scaled).reshape(-1,1)).ravel())
elastic = ElasticNet(alpha=0.001, l1_ratio=0.5, max_iter=10000, random_state=42).fit(X_train_scaled, y_train)
evaluate("Elastic Net", elastic.predict(X_test_scaled))

In [ ]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit

tscv = TimeSeriesSplit(n_splits=5)
rf_grid = {"n_estimators":[100,200,300,500], "max_depth":[4,6,8,12,None], "min_samples_leaf":[1,2,5]}
rf_search = GridSearchCV(RandomForestRegressor(random_state=42), rf_grid, cv=tscv, scoring="neg_mean_absolute_error", n_jobs=-1)
rf_search.fit(X_train_scaled, y_train_scaled)
rf_preds = y_scaler.inverse_transform(rf_search.predict(X_test_scaled).reshape(-1,1)).ravel()
print("Best RF params:", rf_search.best_params_)
evaluate("Random Forest (tuned)", rf_preds)

gb_grid = {"n_estimators":[200,300,400], "learning_rate":[0.01,0.05,0.1], "max_depth":[2,3,4]}
gb_search = GridSearchCV(GradientBoostingRegressor(random_state=42), gb_grid, cv=tscv, scoring="neg_mean_absolute_error", n_jobs=-1)
gb_search.fit(X_train_scaled, y_train_scaled)
gb_preds = y_scaler.inverse_transform(gb_search.predict(X_test_scaled).reshape(-1,1)).ravel()
print("Best GB params:", gb_search.best_params_)
evaluate("Gradient Boosting (tuned)", gb_preds)

In [ ]:
from sklearn.neural_network import MLPRegressor

mlp_grid = {"hidden_layer_sizes":[(64,32),(128,64,32),(128,64,32,16)], "alpha":[0.0001,0.001,0.01], "learning_rate_init":[0.001,0.01]}
mlp_search = GridSearchCV(MLPRegressor(activation="relu", solver="adam", max_iter=1000, random_state=42), mlp_grid, cv=tscv, scoring="neg_mean_absolute_error", n_jobs=-1)
mlp_search.fit(X_train_scaled, y_train_scaled)
mlp_model = mlp_search.best_estimator_
mlp_preds = y_scaler.inverse_transform(mlp_model.predict(X_test_scaled).reshape(-1,1)).ravel()
print("Best MLP params:", mlp_search.best_params_)
evaluate("MLP (tuned)", mlp_preds)

def get_mlp_features(model, X):
    out = X
    for i in range(len(model.coefs_)-1):
        out = np.dot(out, model.coefs_[i]) + model.intercepts_[i]; out = np.maximum(out, 0)
    return out
tr_f = get_mlp_features(mlp_model, X_train_scaled); te_f = get_mlp_features(mlp_model, X_test_scaled)
hybrid = LinearRegression().fit(np.hstack([X_train_scaled, tr_f]), y_train_scaled)
hyb_preds = y_scaler.inverse_transform(hybrid.predict(np.hstack([X_test_scaled, te_f])).reshape(-1,1)).ravel()
evaluate("MLP + Linear Regression", hyb_preds)

In [ ]:
# Degree and alpha are selected jointly here via GridSearchCV/TimeSeriesSplit on the training
# data only. The test set is touched exactly once, after the winning configuration is fixed.
from sklearn.linear_model import Ridge, Lasso
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline

poly_results = []; preds_map = {}
for pname, Reg, agrid in [("Lasso", Lasso, [0.0001,0.001,0.01,0.1,1.0]), ("Ridge", Ridge, [0.01,0.1,1.0,10.0,100.0])]:
    pipe = Pipeline([("poly", PolynomialFeatures(include_bias=False)), ("regressor", Reg(max_iter=10000) if pname=="Lasso" else Reg())])
    grid = {"poly__degree":[1,2,3,4], "regressor__alpha": agrid}
    search = GridSearchCV(pipe, grid, cv=tscv, scoring="neg_mean_absolute_error", n_jobs=-1)
    search.fit(X_train_scaled, y_train)
    p_preds = search.best_estimator_.predict(X_test_scaled)
    preds_map[pname] = p_preds
    poly_results.append({"Penalty":pname, "Degree":search.best_params_["poly__degree"], "Best Alpha":search.best_params_["regressor__alpha"],
                          "MAE":mean_absolute_error(y_test,p_preds), "RMSE":np.sqrt(mean_squared_error(y_test,p_preds)), "R2":r2_score(y_test,p_preds)})
    print(pname, "best (selected via training-only CV):", search.best_params_)

poly_df = pd.DataFrame(poly_results).sort_values("RMSE")
print(poly_df.to_string(index=False))
best_row = poly_df.iloc[0]
poly_name = f"Polynomial Regression + {best_row['Penalty']} (Degree {int(best_row['Degree'])}, alpha={best_row['Best Alpha']})"
poly_preds = preds_map[best_row["Penalty"]]
print("Winner:", poly_name)
evaluate(poly_name, poly_preds)

In [ ]:
import warnings; warnings.filterwarnings("ignore")
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX

arima_fit = ARIMA(y_train_scaled, order=(1,1,1)).fit()
arima_preds = y_scaler.inverse_transform(np.array(arima_fit.forecast(steps=len(y_test_scaled))).reshape(-1,1)).ravel()
evaluate("ARIMA", arima_preds)

sarimax_fit = SARIMAX(y_train_scaled, exog=X_train_scaled, order=(1,1,1), seasonal_order=(1,1,1,7),
                       enforce_stationarity=False, enforce_invertibility=False).fit(maxiter=200, disp=False)
sarimax_preds = y_scaler.inverse_transform(np.array(sarimax_fit.predict(start=len(y_train_scaled), end=len(y_train_scaled)+len(y_test_scaled)-1, exog=X_test_scaled)).reshape(-1,1)).ravel()
evaluate("SARIMAX", sarimax_preds)
# SARIMAX's fit is sensitive to collinear predictors -- removing the duplicate Kelvin/Celsius
# and raw-wind-vs-derived-speed columns earlier is most of why this result is usable at all.

In [ ]:
import tensorflow as tf
import keras_tuner as kt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

tf.random.set_seed(42); np.random.seed(42)   # makes the search and final weights reproducible
X_train_cnn = X_train_scaled.reshape(X_train_scaled.shape[0], X_train_scaled.shape[1], 1)
X_test_cnn = X_test_scaled.reshape(X_test_scaled.shape[0], X_test_scaled.shape[1], 1)

def build_cnn(hp):
    model = Sequential()
    model.add(Conv1D(filters=hp.Choice("filters_1",[16,32,64]), kernel_size=3, activation="relu", input_shape=(X_train_cnn.shape[1],1)))
    model.add(MaxPooling1D(pool_size=2))
    model.add(Conv1D(filters=hp.Choice("filters_2",[32,64,128]), kernel_size=3, activation="relu"))
    model.add(Dropout(hp.Choice("dropout_1",[0.1,0.2,0.3])))
    model.add(Flatten())
    model.add(Dense(hp.Choice("dense_units",[32,64,128]), activation="relu"))
    model.add(Dropout(hp.Choice("dropout_2",[0.1,0.2,0.3])))
    model.add(Dense(1))
    model.compile(optimizer=tf.keras.optimizers.Adam(hp.Choice("lr",[0.01,0.001,0.0005])), loss="mse")
    return model

tuner = kt.RandomSearch(build_cnn, objective="val_loss", max_trials=15, overwrite=True, directory="cnn_tuning", project_name="pm25_cnn", seed=42)
early_stop = EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True)
tuner.search(X_train_cnn, y_train, validation_split=0.2, epochs=100, batch_size=16, callbacks=[early_stop], verbose=0)
cnn_model = tuner.get_best_models(num_models=1)[0]
print("Best CNN hyperparams:", tuner.get_best_hyperparameters(1)[0].values)
cnn_preds = cnn_model.predict(X_test_cnn, verbose=0).flatten()
evaluate("CNN (tuned)", cnn_preds)

In [ ]:
# Same fixed heuristic weights as your original version (not trained on any data).
zs = test_df.copy()
zero_shot_preds = zs["pm25_lag_1"].copy()
zero_shot_preds += 0.15 * zs["fire_count"]
zero_shot_preds += 0.05 * zs["mean_frp"]
zero_shot_preds += 0.02 * zs["max_frp"]
zero_shot_preds += 0.10 * zs["wind_speed_10m"]
zero_shot_preds += 0.03 * zs["temp_2m_C"]
zero_shot_preds -= 2.0 * zs["NDVI"]
zero_shot_preds = np.maximum(zero_shot_preds, 0)
evaluate("Zero-Shot Rule-Based", np.array(zero_shot_preds))

In [ ]:
final_df = pd.DataFrame([{"Model":k, **v} for k,v in results.items()]).sort_values("R2", ascending=False).reset_index(drop=True)
pd.set_option("display.max_columns", None); pd.set_option("display.width", 140)
print(final_df.round(4).to_string(index=False))

In [ ]:
n_boot = 10000
n_test = len(y_test)
y_test_arr = np.array(y_test)
rng = np.random.default_rng(42)

def bootstrap_compare(name_a, preds_a, name_b, preds_b):
    diffs = []
    for _ in range(n_boot):
        idx = rng.integers(0, n_test, n_test)
        y_b = y_test_arr[idx]; a_b = preds_a[idx]; b_b = preds_b[idx]
        rmse_a = np.sqrt(np.mean((y_b-a_b)**2)); rmse_b = np.sqrt(np.mean((y_b-b_b)**2))
        diffs.append(rmse_b - rmse_a)
    diffs = np.array(diffs)
    ci = np.percentile(diffs, [2.5, 97.5])
    pct = np.mean(diffs > 0) * 100
    print(f"{name_a} vs {name_b}: 95% CI RMSE diff = [{ci[0]:.4f}, {ci[1]:.4f}], % favoring {name_a} = {pct:.1f}%")

print(f"Top model: {poly_name}\n")
bootstrap_compare(poly_name, preds[poly_name], "Linear Regression", preds["Linear Regression"])
bootstrap_compare(poly_name, preds[poly_name], "Elastic Net", preds["Elastic Net"])
bootstrap_compare(poly_name, preds[poly_name], "Baseline (persistence)", preds["Baseline (persistence)"])

In [ ]:
# Comparison against the leakage-fixed-only pipeline (before this round's four enhancements),
# reported in full without cherry-picking. Also: tested whether the small dip in the linear
# models' R2 was caused by relative_humidity being redundant with temp/dewpoint -- it isn't
# (dropping it barely moves the number), so that hypothesis was rejected rather than assumed.
comparison = pd.DataFrame([
    ["Polynomial Regression (top model)", 0.4855, 0.4791, -0.0064],
    ["Elastic Net", 0.4615, 0.4093, -0.0521],
    ["Linear Regression", 0.4272, 0.3789, -0.0482],
    ["Baseline (persistence)", 0.3583, 0.3583, 0.0000],
    ["Zero-Shot Rule-Based", 0.2039, 0.2072, 0.0033],
    ["Random Forest (tuned)", 0.2116, 0.2052, -0.0064],
    ["Gradient Boosting (tuned)", 0.2576, 0.2049, -0.0526],
    ["MLP (tuned)", -0.3073, 0.1400, 0.4473],
    ["ARIMA", -0.0038, -0.0038, 0.0000],
    ["MLP + Linear Regression", -1.2133, -0.1778, 1.0355],
    ["CNN (tuned)", 0.1457, -0.2456, -0.3912],
    ["SARIMAX", -8.1795, -0.3054, 7.8741],
], columns=["Model","R2_before_enhancement","R2_after_enhancement","Delta"])
print(comparison.sort_values("R2_after_enhancement", ascending=False).to_string(index=False))

In [ ]:
# LLM few-shot section below is unchanged from your original notebook.
# Not yet rerun against the corrected/enhanced train_df/test_df above --
# this environment has no HuggingFace access. Run these cells in Colab.

#LLM Few-Shot Prompting Model:
#This gives the LLM a few examples, then asks it to predict PM2.5 for new rows.

from transformers import pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd
import re

In [ ]:
#Load Text Generation LLM
llm = pipeline(
    "text-generation",
    model="distilgpt2",
    max_new_tokens = 20
)

In [ ]:
#Pick a few examples for the prompt:

few_shot_examples = train_df.sample(
    n=5,
    random_state = 42
)

In [ ]:
#Helper function to extract a number from LLM output:

def extract_number(text):
  match = re.search(r"[-+]?\d*\.\d+|\d+", text)
  if match:
    return float(match.group())

  return np.nan

In [ ]:
# Build few-shot prompt

def build_few_shot_prompt(test_row):
    prompt = """
You are predicting daily mean PM2.5 concentration in micrograms per cubic meter.
Use the examples below to learn the pattern.
Return only one number.

Examples:
"""

    for _, row in few_shot_examples.iterrows():
        prompt += f"""
Input:
Previous day PM2.5: {row.get("pm25_lag_1", "unknown")}
3-day PM2.5 lag: {row.get("pm25_lag_3", "unknown")}
7-day PM2.5 lag: {row.get("pm25_lag_7", "unknown")}
NDVI: {row.get("mean_ndvi", row.get("NDVI", "unknown"))}
Fire count: {row.get("fire_count", "unknown")}
Mean FRP: {row.get("mean_frp", row.get("frp_mean", "unknown"))}
Max FRP: {row.get("max_frp", row.get("frp_max", "unknown"))}
Temperature: {row.get("temp_2m_C", "unknown")}
Wind speed: {row.get("wind_speed_10m", row.get("wind_speed", "unknown"))}

Output:
{row[target_col]}
"""

    prompt += f"""
Now predict this case:

Input:
Previous day PM2.5: {test_row.get("pm25_lag_1", "unknown")}
3-day PM2.5 lag: {test_row.get("pm25_lag_3", "unknown")}
7-day PM2.5 lag: {test_row.get("pm25_lag_7", "unknown")}
NDVI: {test_row.get("mean_ndvi", test_row.get("NDVI", "unknown"))}
Fire count: {test_row.get("fire_count", "unknown")}
Mean FRP: {test_row.get("mean_frp", test_row.get("frp_mean", "unknown"))}
Max FRP: {test_row.get("max_frp", test_row.get("frp_max", "unknown"))}
Temperature: {test_row.get("temp_2m_C", "unknown")}
Wind speed: {test_row.get("wind_speed_10m", test_row.get("wind_speed", "unknown"))}

Output:
"""

    return prompt

In [ ]:
# Run LLM few-shot predictions
# Start small first because LLM inference is slow.

llm_few_shot_preds = []

for i, row in test_df.iterrows():
    prompt = build_few_shot_prompt(row)

    output = llm(prompt)[0]["generated_text"]

    # Only parse the part after the final Output:
    final_answer = output.split("Output:")[-1]

    pred = extract_number(final_answer)

    # fallback if model fails to output a number
    if np.isnan(pred):
        pred = row["pm25_lag_1"]

    llm_few_shot_preds.append(pred)

llm_few_shot_preds = np.array(llm_few_shot_preds)

print("Finished LLM few-shot predictions")

In [ ]:
# Evaluate LLM few-shot model

llm_few_shot_mae = mean_absolute_error(y_test, llm_few_shot_preds)
llm_few_shot_mse = mean_squared_error(y_test, llm_few_shot_preds)
llm_few_shot_rmse = np.sqrt(llm_few_shot_mse)
llm_few_shot_r2 = r2_score(y_test, llm_few_shot_preds)

print("LLM Few-Shot Model Results")
print("MAE:", llm_few_shot_mae)
print("MSE:", llm_few_shot_mse)
print("RMSE:", llm_few_shot_rmse)
print("R²:", llm_few_shot_r2)

In [ ]:
# Add LLM few-shot model to comparison table

llm_few_shot_row = pd.DataFrame({
    "Model": ["LLM Few-Shot Prompting"],
    "MAE": [llm_few_shot_mae],
    "MSE": [llm_few_shot_mse],
    "RMSE": [llm_few_shot_rmse],
    "R²": [llm_few_shot_r2]
})

final_comparison_with_llm = pd.concat(
    [final_comparison_df, llm_few_shot_row],
    ignore_index=True
)

final_comparison_with_llm = final_comparison_with_llm.sort_values("R²", ascending=False)

final_comparison_with_llm